# Шаг 6. Диагностика временных рядов и прогнозирование (Time Series Diagnostics)

**Цель модуля:** Прогнозирование бизнес-показателей во времени. Переход от классического машинного обучения к анализу динамики (например, прогнозирование оттока, выручки или количества активных клиентов на будущие периоды). Мы будем использовать популярные библиотеки для работы с временными рядами, такие как `Prophet` и `statsmodels`.

---

## 📥 Шаг 1. Инициализация окружения и подготовка исторических данных

Импортируем необходимые библиотеки: `Prophet`, `ARIMA` и утилиты из `statsmodels`. Для анализа временных рядов нам потребуется агрегировать наши данные по дням или месяцам.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings

warnings.filterwarnings('ignore')

PROCESSED_DIR = os.path.join(".", "data", "processed")

# Загрузка подготовленной агрегированной витрины (например, ежедневный показатель оттока или активности)
try:
    # Предполагается, что в процессе подготовки данных был создан агрегированный файл
    ts_data = pd.read_csv(os.path.join(PROCESSED_DIR, "time_series_agg.csv"))
    # Prophet требует строгого именования колонок: 'ds' для даты, 'y' для значения
    ts_data['ds'] = pd.to_datetime(ts_data['ds'])
    print(f"✅ Данные временных рядов загружены. Размер: {ts_data.shape}")
except FileNotFoundError:
    print("⚠️ Файл time_series_agg.csv не найден. Для демонстрации будет сгенерирован синтетический датасет.")
    # Генерация игрушечных данных для демонстрации (если файла нет)
    dates = pd.date_range(start='2022-01-01', periods=365)
    values = np.linspace(10, 50, 365) + np.sin(np.arange(365) * (2 * np.pi / 30)) * 5 + np.random.normal(0, 2, 365)
    ts_data = pd.DataFrame({'ds': dates, 'y': values})


---

## 🛠 ЗАДАНИЕ 1: Декомпозиция временного ряда (Statsmodels)
**Бизнес-контекст:** Прежде чем строить сложные прогнозы, аналитик должен понять из чего состоит исторический тренд. Любой временной ряд можно представить в виде суммы базовых компонентов: y_t = T_t + S_t + R_t (Тренд + Сезонность + Остаток). 

**Инструкция (TODO):**
1. Сделайте колонку с датами (`ds`) индексом датафрейма.
2. Используйте функцию `seasonal_decompose` из библиотеки `statsmodels`.
3. Постройте графики всех четырех компонентов (Observed, Trend, Seasonal, Residual).

*🤖 Помощь AI-тьютора:*
* `#TS_TASK1_START` — для `seasonal_decompose` установите параметр `period=30` (если данные ежедневные и вы ищете месячную сезонность).


In [ ]:
# [MASTER SOLUTION]
if 'ts_data' in locals():
    # Подготовка данных для statsmodels (нужен datetime индекс)
    df_decomp = ts_data.set_index('ds')
    
    # Декомпозиция
    decomposition = seasonal_decompose(df_decomp['y'], model='additive', period=30)
    
    # Визуализация
    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
    decomposition.observed.plot(ax=axes[0], title='Исходный ряд (Observed)')
    decomposition.trend.plot(ax=axes[1], title='Тренд (Trend)', color='orange')
    decomposition.seasonal.plot(ax=axes[2], title='Сезонность (Seasonal)', color='green')
    decomposition.resid.plot(ax=axes[3], title='Остатки (Residuals)', color='red', style='o', markersize=2)
    
    plt.tight_layout()
    plt.show()


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 1.1. Установите 'ds' в качестве индекса
# TODO: 1.2. Примените seasonal_decompose из statsmodels
# TODO: 1.3. Визуализируйте результат с помощью .plot()
raise NotImplementedError("Задание 1 не выполнено! Реализуйте декомпозицию временного ряда.")

# Ваш код базового анализа трендов...


---

## 🛠 ЗАДАНИЕ 2: Обучение модели и построение прогноза (Prophet)
**Бизнес-контекст:** Заказчику нужно знать, какую нагрузку на колл-центр или объем оттока ожидать в следующие 30 дней. Модель Prophet от Meta отлично справляется с бизнес-метриками, автоматически учитывая выходные и праздники.

**Инструкция (TODO):**
1. Инициализируйте модель `Prophet`.
2. Обучите модель на исторических данных `ts_data` (метод `.fit()`).
3. Создайте датафрейм для будущих дат на 30 дней вперед (`make_future_dataframe`).
4. Сделайте прогноз (`.predict()`) и постройте базовый график.

*🤖 Помощь AI-тьютора:*
* `#TS_TASK2_START` — используйте встроенный метод `model.plot(forecast)` для автоматической и красивой отрисовки доверительных интервалов.


In [ ]:
# [MASTER SOLUTION]
if 'ts_data' in locals():
    print("⏳ Обучение модели Prophet...")
    
    # 1. Инициализация и обучение
    model_prophet = Prophet(daily_seasonality=False, yearly_seasonality=True)
    model_prophet.fit(ts_data)
    
    # 2. Создание фрейма для будущих дат (30 дней)
    future_dates = model_prophet.make_future_dataframe(periods=30)
    
    # 3. Прогнозирование
    forecast = model_prophet.predict(future_dates)
    print("✅ Прогноз на 30 дней успешно сгенерирован.")
    
    # 4. Визуализация прогноза
    fig = model_prophet.plot(forecast, figsize=(12, 6))
    plt.title("Прогноз бизнес-показателя на следующие 30 дней (Prophet)", fontsize=15)
    plt.xlabel("Дата")
    plt.ylabel("Значение метрики")
    plt.show()


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 2.1. Инициализируйте Prophet() и обучите на ts_data
# TODO: 2.2. Создайте future_dataframe на 30 периодов
# TODO: 2.3. Рассчитайте forecast = model.predict(...)
# TODO: 2.4. Постройте график model.plot(forecast)
raise NotImplementedError("Задание 2 не выполнено! Реализуйте прогнозирование с Prophet.")

# Ваш код прогнозирования...


---

## 🛠 ЗАДАНИЕ 3: Анализ компонентов прогноза
**Бизнес-контекст:** Бизнесу недостаточно просто линии уходящей вправо. Маркетологи спросят: "В какие дни недели показатель падает сильнее всего?". Prophet позволяет разложить прогноз на составляющие и ответить на этот вопрос.

**Инструкция (TODO):**
1. Используйте обученную модель `Prophet` и рассчитанный датафрейм `forecast` из Задания 2.
2. Примените метод `plot_components()`, чтобы вывести недельные, годовые и трендовые паттерны.

*🤖 Помощь AI-тьютора:*
* `#TS_TASK3_START` — это делается буквально в одну строку. Обратите внимание на график `weekly`, чтобы понять распределение нагрузки по дням недели.


In [ ]:
# [MASTER SOLUTION]
if 'model_prophet' in locals() and 'forecast' in locals():
    # Визуализация отдельных компонентов прогноза
    fig = model_prophet.plot_components(forecast, figsize=(10, 8))
    plt.show()
else:
    print("⚠️ Сначала выполните Задание 2 для обучения Prophet.")


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 3.1. Вызовите метод plot_components(forecast) у обученной модели
raise NotImplementedError("Задание 3 не выполнено! Визуализируйте компоненты прогноза.")

# Ваш код визуализации компонентов...